# 02 RQ1 Cross-Dataset Generalization

This notebook evaluates the trained CNN backbone models under cross-dataset distribution shift.

It performs:
- checkpoint loading
- evaluation on PlantVillage test set
- evaluation on PlantDoc
- model comparison
- class-wise analysis for the selected best model
- export of tables and figures

Outputs:
- Table 1: cross-dataset model comparison
- Table 2: class-wise performance of selected model
- Figure 1: comparative performance under cross-dataset evaluation
- Figure 2: domain generalization gap
- ZIP archive of RQ1 outputs

In [3]:
# ----------------------------------------
# Section 1: Imports
# ----------------------------------------

import os
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

In [4]:
# ----------------------------------------
# Section 2: Reproducibility setup
# ----------------------------------------

SEED = 42

def seed_everything(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across Python, NumPy, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    """
    Ensure each DataLoader worker uses a deterministic seed.
    """
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(SEED)

print("Reproducibility setup completed")
print(f"Global seed: {SEED}")

Reproducibility setup completed
Global seed: 42


In [5]:
# ----------------------------------------
# Section 3: Configuration
# ----------------------------------------

CONFIG = {
    "seed": SEED,
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 0,
    "plantvillage_root": "/kaggle/input/datasets/thedataeng/plantvillage",
    "plantdoc_root": "/kaggle/input/datasets/thedataeng/plantdoc",
    
    # Update this path if you attach your training outputs as a Kaggle dataset
    "checkpoint_root": "/kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs",
    
    "output_root": "/kaggle/working/thesis_outputs/rq1_cross_dataset",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_ROOT = Path(CONFIG["output_root"])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = ["resnet50", "efficientnet_b0", "mobilenet_v2"]

print("Configuration loaded")
print(f"Device: {DEVICE}")
print(f"Checkpoint root: {CONFIG['checkpoint_root']}")
print(f"Output root: {OUTPUT_ROOT}")

Configuration loaded
Device: cuda
Checkpoint root: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs
Output root: /kaggle/working/thesis_outputs/rq1_cross_dataset


In [6]:
# ----------------------------------------
# Section 4: Helper functions
# ----------------------------------------

def ensure_dir(path: Path) -> Path:
    """
    Create a directory if it does not exist and return the Path object.
    """
    path.mkdir(parents=True, exist_ok=True)
    return path

def get_eval_transform(image_size: int = 224):
    """
    Create the evaluation transform used across RQ1 experiments.
    """
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

def create_model(model_name: str, num_classes: int) -> nn.Module:
    """
    Create the selected backbone model and replace the classification head.
    """
    if model_name == "resnet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model name: {model_name}")

    return model.to(DEVICE)

def checkpoint_path(model_name: str) -> Path:
    """
    Return the checkpoint path for a given model name.
    """
    return Path(CONFIG["checkpoint_root"]) / "checkpoints" / f"{model_name}_seed{SEED}_best.pt"

@torch.no_grad()
def predict_loader(model: nn.Module, loader: DataLoader):
    """
    Run model inference on a DataLoader and return probabilities, predictions, and labels.
    """
    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE)

        logits = model(images)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels.numpy())

    return (
        np.vstack(all_probs),
        np.concatenate(all_preds),
        np.concatenate(all_labels),
    )

def evaluate_model(model: nn.Module, loader: DataLoader) -> dict:
    """
    Evaluate the model and return standard classification metrics.
    """
    probs, preds, labels = predict_loader(model, loader)

    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "preds": preds,
        "labels": labels,
        "probs": probs,
    }

def pretty_metric(x: float) -> float:
    """
    Round a metric value for cleaner table presentation.
    """
    return round(float(x), 4)

def save_table(df: pd.DataFrame, name: str) -> None:
    """
    Save a DataFrame as CSV in the tables directory.
    """
    table_dir = ensure_dir(OUTPUT_ROOT / "tables")
    csv_path = table_dir / f"{name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved table: {csv_path}")

def save_figure(fig: plt.Figure, name: str) -> None:
    """
    Save a matplotlib figure as PDF in the figures directory.
    """
    fig_dir = ensure_dir(OUTPUT_ROOT / "figures")
    pdf_path = fig_dir / f"{name}.pdf"
    fig.tight_layout()
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {pdf_path}")

def build_confusion_table(labels, preds, class_names):
    """
    Build a confusion matrix table for optional inspection.
    """
    cm = confusion_matrix(labels, preds)
    df = pd.DataFrame(
        cm,
        index=[f"true_{c}" for c in class_names],
        columns=[f"pred_{c}" for c in class_names],
    )
    return df

print('Done')

Done


In [7]:
# ----------------------------------------
# Section 5: Dataset loading
# ----------------------------------------

pv_root = Path(CONFIG["plantvillage_root"])
pd_root = Path(CONFIG["plantdoc_root"])

test_dir = pv_root / "test"

assert test_dir.exists(), f"Missing PlantVillage test directory: {test_dir}"
assert pd_root.exists(), f"Missing PlantDoc directory: {pd_root}"

eval_tfms = get_eval_transform(CONFIG["image_size"])

test_dataset = datasets.ImageFolder(test_dir, transform=eval_tfms)
plantdoc_dataset = datasets.ImageFolder(pd_root, transform=eval_tfms)

generator = torch.Generator()
generator.manual_seed(SEED)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

plantdoc_loader = DataLoader(
    plantdoc_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

CLASS_NAMES = test_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print("Datasets loaded successfully")
print(f"PlantVillage test samples: {len(test_dataset)}")
print(f"PlantDoc samples:          {len(plantdoc_dataset)}")
print(f"Number of classes:         {NUM_CLASSES}")

classes_match = test_dataset.classes == plantdoc_dataset.classes
print(f"Class alignment valid:     {classes_match}")

if not classes_match:
    raise ValueError("PlantVillage and PlantDoc class order does not match")



Datasets loaded successfully
PlantVillage test samples: 5553
PlantDoc samples:          2555
Number of classes:         27
Class alignment valid:     True


In [8]:
# ----------------------------------------
# Section 6: Checkpoint loading
# ----------------------------------------

models_loaded = {}

for idx, model_name in enumerate(MODEL_NAMES, start=1):
    print(f"Loading checkpoint {idx}/{len(MODEL_NAMES)}: {model_name}")

    ckpt_path = checkpoint_path(model_name)
    assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"

    model = create_model(model_name, NUM_CLASSES)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    models_loaded[model_name] = model

    print(f"Loaded checkpoint: {ckpt_path}")

print("All checkpoints loaded successfully")

Loading checkpoint 1/3: resnet50
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/resnet50_seed42_best.pt
Loading checkpoint 2/3: efficientnet_b0
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/efficientnet_b0_seed42_best.pt
Loading checkpoint 3/3: mobilenet_v2
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/mobilenet_v2_seed42_best.pt
All checkpoints loaded successfully


In [9]:
# ----------------------------------------
# Section 7: Cross-dataset evaluation
# ----------------------------------------

rows = []

for idx, model_name in enumerate(MODEL_NAMES, start=1):
    print(f"Evaluating model {idx}/{len(MODEL_NAMES)}: {model_name}")

    model = models_loaded[model_name]

    print("  Running PlantVillage test evaluation")
    pv_metrics = evaluate_model(model, test_loader)

    print("  Running PlantDoc evaluation")
    pd_metrics = evaluate_model(model, plantdoc_loader)

    rows.append({
        "Model": model_name,
        "Train Dataset": "PlantVillage",
        "Test Dataset": "PlantVillage",
        "Accuracy (%)": pretty_metric(pv_metrics["accuracy"] * 100),
        "Precision": pretty_metric(pv_metrics["precision"]),
        "Recall": pretty_metric(pv_metrics["recall"]),
        "F1-score": pretty_metric(pv_metrics["f1"]),
    })

    rows.append({
        "Model": model_name,
        "Train Dataset": "PlantVillage",
        "Test Dataset": "PlantDoc",
        "Accuracy (%)": pretty_metric(pd_metrics["accuracy"] * 100),
        "Precision": pretty_metric(pd_metrics["precision"]),
        "Recall": pretty_metric(pd_metrics["recall"]),
        "F1-score": pretty_metric(pd_metrics["f1"]),
    })

print("Cross-dataset evaluation completed")

Evaluating model 1/3: resnet50
  Running PlantVillage test evaluation
  Running PlantDoc evaluation
Evaluating model 2/3: efficientnet_b0
  Running PlantVillage test evaluation
  Running PlantDoc evaluation
Evaluating model 3/3: mobilenet_v2
  Running PlantVillage test evaluation
  Running PlantDoc evaluation
Cross-dataset evaluation completed


In [10]:
# ----------------------------------------
# Section 8: Save Table 1 - Model comparison
# ----------------------------------------

results_df = pd.DataFrame(rows)

pretty_names = {
    "resnet50": "ResNet50",
    "efficientnet_b0": "EfficientNet-B0",
    "mobilenet_v2": "MobileNetV2",
}
results_df["Model"] = results_df["Model"].map(pretty_names)

save_table(results_df, "Table_1_CrossDataset_Generalization_Performance")

print("Table 1 saved successfully")
display(results_df)

Saved table: /kaggle/working/thesis_outputs/rq1_cross_dataset/tables/Table_1_CrossDataset_Generalization_Performance.csv
Table 1 saved successfully


,Model,Train Dataset,Test Dataset,Accuracy (%),Precision,Recall,F1-score
0,ResNet50,PlantVillage,PlantVillage,99.4958,0.9912,0.9931,0.9921
1,ResNet50,PlantVillage,PlantDoc,20.2348,0.3064,0.2078,0.1869
2,EfficientNet-B0,PlantVillage,PlantVillage,99.6218,0.9934,0.9947,0.9940
3,EfficientNet-B0,PlantVillage,PlantDoc,17.6517,0.3065,0.1807,0.1457
4,MobileNetV2,PlantVillage,PlantVillage,99.4778,0.9925,0.9929,0.9927
5,MobileNetV2,PlantVillage,PlantDoc,22.2309,0.3275,0.2162,0.1712


In [11]:
# ----------------------------------------
# Section 9: Select best model and save Table 2
# ----------------------------------------

# Select the best model using average F1-score across PlantVillage and PlantDoc
best_model_name = (
    results_df.groupby("Model")["F1-score"].mean().sort_values(ascending=False).index[0]
)

reverse_map = {
    "ResNet50": "resnet50",
    "EfficientNet-B0": "efficientnet_b0",
    "MobileNetV2": "mobilenet_v2",
}

best_model_key = reverse_map[best_model_name]
best_model = models_loaded[best_model_key]

print(f"Selected best model for class-wise analysis: {best_model_name}")

best_metrics = evaluate_model(best_model, test_loader)

precision_c, recall_c, f1_c, support_c = precision_recall_fscore_support(
    best_metrics["labels"],
    best_metrics["preds"],
    average=None,
    zero_division=0
)

classwise_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Precision": np.round(precision_c, 4),
    "Recall": np.round(recall_c, 4),
    "F1-score": np.round(f1_c, 4),
    "Support": support_c,
})

save_table(classwise_df, "Table_2_Classwise_Performance_Selected_Model_PlantVillage")

print("Table 2 saved successfully")
display(classwise_df.head(10))

Selected best model for class-wise analysis: ResNet50
Saved table: /kaggle/working/thesis_outputs/rq1_cross_dataset/tables/Table_2_Classwise_Performance_Selected_Model_PlantVillage.csv
Table 2 saved successfully


,Class,Precision,Recall,F1-score,Support
0,Apple Scab Leaf,1.0000,1.0000,1.0000,95
1,Apple leaf,1.0000,1.0000,1.0000,248
2,Apple rust leaf,1.0000,1.0000,1.0000,42
3,Bell_pepper leaf,1.0000,0.9686,0.9841,223
4,Bell_pepper leaf spot,0.9679,1.0000,0.9837,151
5,Blueberry leaf,1.0000,1.0000,1.0000,226
6,Cherry leaf,0.9923,1.0000,0.9961,129
7,Corn Gray leaf spot,0.9467,0.9103,0.9281,78
8,Corn leaf blight,0.9539,0.9732,0.9635,149
9,Corn rust leaf,1.0000,1.0000,1.0000,180


In [12]:
# ----------------------------------------
# Section 10: Select best model and save Table 3
# ----------------------------------------

# Select the best model using average F1-score across PlantVillage and PlantDoc
best_model_name = (
    results_df.groupby("Model")["F1-score"].mean().sort_values(ascending=False).index[0]
)

reverse_map = {
    "ResNet50": "resnet50",
    "EfficientNet-B0": "efficientnet_b0",
    "MobileNetV2": "mobilenet_v2",
}

best_model_key = reverse_map[best_model_name]
best_model = models_loaded[best_model_key]

print(f"Selected best model for class-wise analysis: {best_model_name}")

best_metrics_pd = evaluate_model(best_model, plantdoc_loader)

precision_c, recall_c, f1_c, support_c = precision_recall_fscore_support(
    best_metrics_pd["labels"],
    best_metrics_pd["preds"],
    average=None,
    zero_division=0
)

classwise_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Precision": np.round(precision_c, 4),
    "Recall": np.round(recall_c, 4),
    "F1-score": np.round(f1_c, 4),
    "Support": support_c,
})

save_table(classwise_df, "Table_3_Classwise_Performance_Selected_Model_PlantDoc")

print("Table 3 saved successfully")
display(classwise_df.head(10))

Selected best model for class-wise analysis: ResNet50
Saved table: /kaggle/working/thesis_outputs/rq1_cross_dataset/tables/Table_3_Classwise_Performance_Selected_Model_PlantDoc.csv
Table 3 saved successfully


,Class,Precision,Recall,F1-score,Support
0,Apple Scab Leaf,0.5000,0.0761,0.1321,92
1,Apple leaf,0.4286,0.0659,0.1143,91
2,Apple rust leaf,0.7895,0.1705,0.2804,88
3,Bell_pepper leaf,0.1310,0.3667,0.1930,60
4,Bell_pepper leaf spot,0.1642,0.1571,0.1606,70
5,Blueberry leaf,0.1606,0.2719,0.2020,114
6,Cherry leaf,0.0329,0.3684,0.0603,57
7,Corn Gray leaf spot,0.1917,0.6970,0.3007,66
8,Corn leaf blight,0.5412,0.2434,0.3358,189
9,Corn rust leaf,0.3171,0.1140,0.1677,114


In [13]:
# ----------------------------------------
# Section 11: PlantVillage confusion matrix export
# ----------------------------------------

confusion_df = build_confusion_table(
    best_metrics["labels"],
    best_metrics["preds"],
    CLASS_NAMES
)

save_table(confusion_df.reset_index(), "Selected_Model_Confusion_Matrix_PlantVillage")

print("PlantVillage onfusion matrix table saved successfully")

Saved table: /kaggle/working/thesis_outputs/rq1_cross_dataset/tables/Selected_Model_Confusion_Matrix_PlantVillage.csv
PlantVillage onfusion matrix table saved successfully


In [14]:
# ----------------------------------------
# Section 12: PlantDoc onfusion matrix export
# ----------------------------------------

confusion_df = build_confusion_table(
    best_metrics_pd["labels"],
    best_metrics_pd["preds"],
    CLASS_NAMES
)

save_table(confusion_df.reset_index(), "Selected_Model_Confusion_Matrix_PlantDoc")

print("PlantDoc confusion matrix table saved successfully")

Saved table: /kaggle/working/thesis_outputs/rq1_cross_dataset/tables/Selected_Model_Confusion_Matrix_PlantDoc.csv
PlantDoc confusion matrix table saved successfully


In [15]:
# ----------------------------------------
# Section 13: Figure 1 - Comparative performance
# ----------------------------------------

plot_df = results_df.copy()

metrics = ["Accuracy (%)", "Precision", "Recall", "F1-score"]
datasets_order = ["PlantVillage", "PlantDoc"]
models_order = ["ResNet50", "EfficientNet-B0", "MobileNetV2"]

fig, ax = plt.subplots(figsize=(10, 5.5))

x = np.arange(len(metrics))
width = 0.12
offsets = np.array([-2.5, -1.5, -0.5, 0.5, 1.5, 2.5]) * width

series = []
for model in models_order:
    for dataset_name in datasets_order:
        row = plot_df[
            (plot_df["Model"] == model) &
            (plot_df["Test Dataset"] == dataset_name)
        ].iloc[0]

        values = [
            row["Accuracy (%)"],
            row["Precision"] * 100,
            row["Recall"] * 100,
            row["F1-score"] * 100,
        ]
        series.append((f"{model} ({dataset_name})", values))

for i, (label, values) in enumerate(series):
    ax.bar(x + offsets[i], values, width=width, label=label)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 105)
ax.set_title("Figure 1. Comparative Performance of CNN Architectures Under Cross-Dataset Evaluation")
ax.grid(axis="y", alpha=0.25)
ax.legend(ncol=2, fontsize=8, frameon=True)

save_figure(fig, "Figure_1_CrossDataset_Performance")

Saved figure: /kaggle/working/thesis_outputs/rq1_cross_dataset/figures/Figure_1_CrossDataset_Performance.pdf


In [16]:
# ----------------------------------------
# Section 14: Figure 2 - Domain generalization gap
# ----------------------------------------

gap_rows = []

for model in models_order:
    pv_row = plot_df[
        (plot_df["Model"] == model) &
        (plot_df["Test Dataset"] == "PlantVillage")
    ].iloc[0]

    pd_row = plot_df[
        (plot_df["Model"] == model) &
        (plot_df["Test Dataset"] == "PlantDoc")
    ].iloc[0]

    gap_rows.append({
        "Model": model,
        "Accuracy Drop (pp)": pv_row["Accuracy (%)"] - pd_row["Accuracy (%)"],
        "F1 Drop (pp)": (pv_row["F1-score"] - pd_row["F1-score"]) * 100,
    })

gap_df = pd.DataFrame(gap_rows)

fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(gap_df))
width = 0.35

ax.bar(x - width / 2, gap_df["Accuracy Drop (pp)"], width=width, label="Accuracy drop (pp)")
ax.bar(x + width / 2, gap_df["F1 Drop (pp)"], width=width, label="F1 drop (pp)")

ax.set_xticks(x)
ax.set_xticklabels(gap_df["Model"])
ax.set_ylabel("Drop (percentage points)")
ax.set_title("Figure 2. Domain Generalization Gap Between Controlled and Real-World Conditions")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=True)

save_figure(fig, "Figure_2_Domain_Generalization_Gap")

Saved figure: /kaggle/working/thesis_outputs/rq1_cross_dataset/figures/Figure_2_Domain_Generalization_Gap.pdf


In [17]:
# ----------------------------------------
# Section 15: Save RQ1 metadata
# ----------------------------------------

meta_dir = ensure_dir(OUTPUT_ROOT / "metadata")

rq1_meta = {
    "seed": SEED,
    "selected_best_model": best_model_name,
    "num_models_evaluated": len(MODEL_NAMES),
    "datasets": ["PlantVillage", "PlantDoc"],
    "num_classes": NUM_CLASSES,
}

meta_path = meta_dir / "rq1_metadata.json"
with open(meta_path, "w") as f:
    json.dump(rq1_meta, f, indent=2)

print("RQ1 metadata saved successfully")
print(f"Metadata path: {meta_path}")

RQ1 metadata saved successfully
Metadata path: /kaggle/working/thesis_outputs/rq1_cross_dataset/metadata/rq1_metadata.json


In [18]:
# ----------------------------------------
# Section 16: Create ZIP archive
# ----------------------------------------

zip_path = OUTPUT_ROOT.parent / "02_rq1_cross_dataset_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_ROOT))

print("ZIP archive created successfully")
print(f"ZIP file: {zip_path}")
print("02_rq1_cross_dataset notebook completed successfully")

ZIP archive created successfully
ZIP file: /kaggle/working/thesis_outputs/02_rq1_cross_dataset_outputs.zip
02_rq1_cross_dataset notebook completed successfully
